# Deer detector: fine-tune YOLO

Run from this folder with the packages in `requirements.txt` installed. This experiment trains on the decoy training split plus converted BAMBI matched RGB Flight 104, then selects and evaluates the model on the unchanged decoy validation split (see `data_bambi_decoy.yaml`).

In [ ]:
from pathlib import Path

EPOCHS = 300
MODEL = "yolo11s.pt"   # yolo11n/s/m/l/x.pt
IMGSZ = 1280           # smallest deer are ~20-40 px at this size, keep it large
PATIENCE = 50          # stop after this many epochs without improvement
IOU = 0.45             # duplicate-box cutoff for val/predict only (Ultralytics default 0.7)
SEED = 0               # record the data/order seed explicitly for reproducibility

ROOT = Path.cwd()
DECOY_DATA = ROOT / "dataset"
BAMBI_DATA = (ROOT / "../../data/bambi_yolo").resolve()
DATA_CONFIG = ROOT / "data_bambi_decoy.yaml"
DATASETS_TO_CHECK = (("decoy", DECOY_DATA), ("BAMBI", BAMBI_DATA))
VALIDATION_CONFIGS = (("decoy proxy val", DECOY_DATA / "data.yaml"),)
RUN_NAME = "yolo11s_bambi104_decoy_v1"      # use a new name for every run
RUN_DIR = ROOT / "runs" / RUN_NAME

## 1. Check the datasets

In [ ]:
for path in (DATA_CONFIG, *(config for _, config in VALIDATION_CONFIGS)):
    assert path.exists(), f"Missing {path}"
print(DATA_CONFIG.read_text())

for source_name, ds in DATASETS_TO_CHECK:
    for split in ("train", "val"):
        images = sorted((ds / "images" / split).glob("*.jpg"))
        labels = sorted((ds / "labels" / split).glob("*.txt"))
        image_stems = {path.stem for path in images}
        label_stems = {path.stem for path in labels}
        assert image_stems == label_stems, (source_name, split,
                                             sorted(image_stems - label_stems)[:5],
                                             sorted(label_stems - image_stems)[:5])
        boxes = sum(len([line for line in path.read_text().splitlines() if line.strip()]) for path in labels)
        empty = sum(1 for p in labels if not p.read_text().strip())
        print(f"{source_name:12s} {split:5s}: {len(images):3d} images, {len(images) - empty:3d} with deer, "
              f"{empty:3d} negatives, {boxes:3d} boxes")

## 2. Check the GPU

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE, BATCH = 0, -1           # -1 = AutoBatch: largest batch that fits in GPU memory
    print("GPU:", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    DEVICE, BATCH = "cpu", 4
    print("No GPU found: training will run on CPU and be very slow.")

## 3. Train

Trains on the decoy training split plus BAMBI Flight 104. The validation line after each epoch scores the unchanged decoy validation split, so the result can be compared with the decoy-only baseline. `best.pt` is selected from that proxy validation performance. Weights land in `runs/<RUN_NAME>/weights/`.

In [ ]:
from ultralytics import YOLO

assert not RUN_DIR.exists(), f"{RUN_DIR} already exists: change RUN_NAME so it is not overwritten"

model = YOLO(MODEL)
model.train(data=str(DATA_CONFIG), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
            patience=PATIENCE, seed=SEED, project=str(ROOT / "runs"), name=RUN_NAME)

BEST = RUN_DIR / "weights" / "best.pt"
assert BEST.exists(), f"No trained weights at {BEST}"
print("Best weights:", BEST)

## 4. Training curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

res = pd.read_csv(RUN_DIR / "results.csv")
res.columns = [c.strip() for c in res.columns]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for c in ("train/box_loss", "val/box_loss"):
    axes[0].plot(res["epoch"], res[c], label=c)
axes[0].set_title("Box loss"); axes[0].legend()
for c in ("metrics/precision(B)", "metrics/recall(B)"):
    axes[1].plot(res["epoch"], res[c], label=c.split("/")[1])
axes[1].set_title("Precision / recall"); axes[1].legend()
for c in ("metrics/mAP50(B)", "metrics/mAP50-95(B)"):
    axes[2].plot(res["epoch"], res[c], label=c.split("/")[1])
axes[2].set_title("mAP"); axes[2].legend()
for ax in axes: ax.set_xlabel("epoch")
plt.tight_layout(); plt.show()
best = res.loc[res["metrics/mAP50-95(B)"].idxmax()]  # same metric best.pt is picked by
print(f"Best epoch {int(best['epoch'])}: mAP50-95={best['metrics/mAP50-95(B)']:.3f}  mAP50={best['metrics/mAP50(B)']:.3f}  "
      f"P={best['metrics/precision(B)']:.3f}  R={best['metrics/recall(B)']:.3f}")

## 5. Validate the best weights

Scores the best weights on the fixed decoy proxy validation set. Add a separate complete-flight BAMBI validation configuration here only after another flight is deliberately held out.

In [ ]:
model = YOLO(str(BEST))
for label, data in VALIDATION_CONFIGS:
    metrics = model.val(data=str(data), imgsz=IMGSZ, iou=IOU, batch=1 if DEVICE == "cpu" else 8, device=DEVICE,
                        project=str(ROOT / "runs"), name="deer_val", exist_ok=True, plots=False, verbose=False, workers=0)
    print(f"{label:17s} precision {metrics.box.mp:.3f} | recall {metrics.box.mr:.3f} | "
          f"mAP50 {metrics.box.map50:.3f} | mAP50-95 {metrics.box.map:.3f}")

## 6. Look at the predictions

Green = labelled deer, red = model prediction with confidence. Negatives (no deer) show
any false alarms. Change `CONF` to see how the threshold trades misses against false alarms.

In [ ]:
from PIL import Image

CONF = 0.25
val_imgs = sorted((DECOY_DATA / "images" / "val").glob("*.jpg"))
preds = [model.predict(str(p), imgsz=IMGSZ, conf=CONF, iou=IOU, device=DEVICE, verbose=False)[0] for p in val_imgs]

def gt_boxes(img_path, w, h):
    lab = DECOY_DATA / "labels" / "val" / (img_path.stem + ".txt")
    out = []
    for line in lab.read_text().split(chr(10)):
        if line.strip():
            _, x, y, bw, bh = map(float, line.split())
            out.append(((x - bw / 2) * w, (y - bh / 2) * h, bw * w, bh * h))
    return out

false_alarm_imgs = 0
n_neg = 0
for p, r in zip(val_imgs, preds):
    w, h = Image.open(p).size
    if not gt_boxes(p, w, h):
        n_neg += 1
        false_alarm_imgs += len(r.boxes) > 0
print(f"Negatives with at least one false alarm at conf={CONF}: {false_alarm_imgs}/{n_neg}")

# Show images with deer first, then negatives that triggered a detection.
show = [(p, r) for p, r in zip(val_imgs, preds) if gt_boxes(p, *Image.open(p).size)]
show += [(p, r) for p, r in zip(val_imgs, preds) if not gt_boxes(p, *Image.open(p).size) and len(r.boxes)]
cols = 3
rows = max(1, -(-len(show) // cols))
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, (p, r) in zip(axes.flat, show):
    img = Image.open(p)
    ax.imshow(img)
    for x, y, bw, bh in gt_boxes(p, *img.size):
        ax.add_patch(plt.Rectangle((x, y), bw, bh, fill=False, color="lime", lw=2))
    for (x1, y1, x2, y2), c in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist()):
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color="red", lw=1.5))
        ax.text(x1, y1, f"{c:.2f}", color="red", fontsize=8, va="bottom")
    ax.set_title(p.stem, fontsize=9)
plt.tight_layout(); plt.show()

## 7. Use the model

```python
from ultralytics import YOLO
model = YOLO("runs/deer_v2/weights/best.pt")
results = model.predict("some_drone_image.jpg", imgsz=1280, conf=0.25, iou=0.45)
```